<a href="https://colab.research.google.com/github/yse-eds-cert/yse-eds-cert-classroom-code-along-notebooks-code_along_notebooks/blob/main/course-4-environmental-analysis/module-13/C4M13L4-tf-idf-topic-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4: TF-IDF and Topic Models for Environmental Text Analysis

This notebook focuses on building interpretable representations for exploration and prediction using TF-IDF and topic modeling techniques.

## Learning Objectives:
- Construct Document-Term Matrices (DTM) and compute TF-IDF weights
- Fit topic models using LDA (Latent Dirichlet Allocation)
- Interpret topic prevalence by source and over time

Let's get started by loading the necessary packages!


In [6]:
# 1. Speeds up compilation by using all available CPU cores
options(Ncpus = parallel::detectCores())

# 2. Installs the critical system dependencies required by tm, tidyverse, and topicmodels
system("sudo apt-get update && sudo apt-get install -y libgsl-dev libxml2-dev libssl-dev libcurl4-openssl-dev")

# 3. Quick, automated bulk installation of the packages
install.packages(c(
  "tidyverse", "tidytext", "topicmodels",
  "SnowballC", "tm", "knitr", "scales",
  "RColorBrewer", "broom"
), repos = "https://r-project.org", quiet = TRUE)

# 4. Verification Check: Attempts to load every package
packages <- c("tidyverse", "tidytext", "topicmodels", "SnowballC", "tm",
              "ggplot2", "dplyr", "stringr", "knitr", "scales", "RColorBrewer", "broom")

verify_load <- sapply(packages, require, character.only = TRUE)

# 5. Print out the final success/failure summary
cat("\n--- INSTALLATION VERIFICATION STATUS ---\n")
print(verify_load)
if(all(verify_load)) {
  cat("\n All libraries installed and loaded successfully!\n")
} else {
  cat("\n WARNING: Some libraries failed to load. Check the list above.\n")
}

Warning message:
“unable to access index for repository https://r-project.org/src/contrib:
  cannot open URL 'https://r-project.org/src/contrib/PACKAGES'”
Warning message:
“packages ‘tidyverse’, ‘tidytext’, ‘topicmodels’, ‘SnowballC’, ‘tm’, ‘knitr’, ‘scales’, ‘RColorBrewer’, ‘broom’ are not available for this version of R

Versions of these packages for your version of R might be available elsewhere,
see the ideas at
https://cran.r-project.org/doc/manuals/r-patched/R-admin.html#Installing-packages”



--- INSTALLATION VERIFICATION STATUS ---
   tidyverse     tidytext  topicmodels    SnowballC           tm      ggplot2 
        TRUE         TRUE         TRUE         TRUE         TRUE         TRUE 
       dplyr      stringr        knitr       scales RColorBrewer        broom 
        TRUE         TRUE         TRUE         TRUE         TRUE         TRUE 

 All libraries installed and loaded successfully!


In [7]:
# Load the required packages
library(tidyverse)     # Data manipulation and visualization
library(tidytext)      # Text mining with tidy data principles
library(topicmodels)   # Topic modeling (LDA)
library(SnowballC)     # Stemming
library(tm)            # Text mining utilities
library(ggplot2)       # Data visualization
library(dplyr)         # Data manipulation
library(stringr)       # String manipulation
library(knitr)         # For nice table output
library(scales)        # For better plot scales
library(RColorBrewer)  # Color palettes
library(broom)         # Tidy model outputs


## Part 1: Data Preparation and Document-Term Matrix Construction

### Loading and Preprocessing the Data

We'll start with the same climate topics dataset, but this time we'll prepare it specifically for TF-IDF analysis and topic modeling.


In [8]:
# Load and clean the climate topics dataset
climate_data <- read_csv("https://github.com/yse-eds-cert/yse-eds-cert-classroom-code-along-notebooks-code_along_notebooks/raw/refs/heads/main/course-4-environmental-analysis/module-13/data/climate_data_course.csv")

# Clean the text data (similar to previous lesson)
climate_clean <- climate_data %>%
  filter(!is.na(text)) %>%
  mutate(text = tolower(text)) %>%
  mutate(text = str_replace_all(text, "https?://\\S+", "")) %>%
  mutate(text = str_replace_all(text, "@\\w+", "")) %>%
  mutate(text = str_replace_all(text, "#", "")) %>%
  mutate(text = str_squish(text)) %>%
  filter(text != "") %>%
  mutate(document_id = row_number())

# Display basic information
cat("Number of documents:", nrow(climate_clean), "\n")
cat("Sample text:", climate_clean$text[1], "\n")


Rows: 2633 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): text
dbl (1): label

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Number of documents: 2633 
Sample text: agriculture needs changes to help save climate and farmers, says national agriculture group | cbc news 


### Tokenization for TF-IDF Analysis

For TF-IDF and topic modeling, we need to be more selective about our tokens to ensure meaningful results.


In [9]:
# Tokenize and filter for topic modeling
climate_tokens <- climate_clean %>%
  unnest_tokens(word, text) %>%
  anti_join(stop_words, by = "word") %>%
  # Remove numbers and single and double characters
  filter(!str_detect(word, "^\\d+")) %>%
  filter(nchar(word) > 2) %>%  # Keep words with 3+ characters
  # Stem the words
  mutate(word = wordStem(word, language = "en")) %>%
  # Remove very rare words (appear in < 5 documents) and very common words (appear in > 50% of documents)
  group_by(word) %>%
  mutate(word_count = n()) %>%
  ungroup() %>%
  filter(word_count >= 5 & word_count <= nrow(climate_clean) * 0.5) %>%
  select(-word_count)

# Check our filtered tokens
cat("Total tokens after filtering:", nrow(climate_tokens), "\n")
cat("Unique words:", length(unique(climate_tokens$word)), "\n")

# Look at most common words
climate_tokens %>%
  count(word, sort = TRUE) %>%
  head(20)


Total tokens after filtering: 22117 
Unique words: 1270 


word,n
<chr>,<int>
climatechang,948
global,414
warm,327
amp,292
peopl,212
cop25,182
climateact,181
world,181
time,158


## Part 2: TF-IDF Analysis

### What is TF-IDF?

**TF-IDF (Term Frequency-Inverse Document Frequency)** is a numerical statistic that reflects how important a word is to a document in a collection of documents.

- **TF (Term Frequency)**: How frequently a term appears in a document
  - Formula: TF(t,d) = (Number of times term t appears in document d) / (Total number of terms in document d)

- **IDF (Inverse Document Frequency)**: How rare or common a term is across all documents
  - Formula: IDF(t) = log(Total number of documents / Number of documents containing term t)

- **TF-IDF**: TF × IDF - gives higher scores to words that are frequent in a document but rare across the corpus
  - Formula: TF-IDF(t,d) = TF(t,d) × IDF(t)

This helps identify words that are particularly characteristic of individual documents.


In [10]:
# Calculate TF-IDF scores
climate_tfidf <- climate_tokens %>%
  count(document_id, word, sort = TRUE) %>%
  bind_tf_idf(word, document_id, n)

# Look at highest TF-IDF scores
climate_tfidf %>%
  arrange(desc(tf_idf)) %>%
  head(20) %>%
  kable(col.names = c("Document ID", "Word", "Count", "TF", "IDF", "TF-IDF"))




| Document ID|Word        | Count| TF|      IDF|   TF-IDF|
|-----------:|:-----------|-----:|--:|--------:|--------:|
|        1540|redistribut |     1|  1| 6.485017| 6.485017|
|          72|author      |     1|  1| 6.261873| 6.261873|
|         466|battl       |     1|  1| 6.261873| 6.261873|
|        1059|index       |     1|  1| 6.261873| 6.261873|
|         616|cult        |     1|  1| 5.925401| 5.925401|
|        1461|scare       |     1|  1| 5.925401| 5.925401|
|        2500|walk        |     1|  1| 5.925401| 5.925401|
|         235|propaganda  |     1|  1| 5.791870| 5.791870|
|        1609|rock        |     1|  1| 5.791870| 5.791870|
|        1726|answer      |     1|  1| 5.674087| 5.674087|
|        2520|transform   |     1|  1| 5.674087| 5.674087|
|        2030|condit      |     1|  1| 5.568726| 5.568726|
|        1003|wors        |     1|  1| 5.473416| 5.473416|
|        1801|adult       |     1|  1| 5.473416| 5.473416|
|        2356|degre       |     1|  1| 5.473416| 5.473

### Some datasets are not great for TF-IDF approaches:
Tweets are so short, we rarely get a TF of greater than 1. TF-IDF tends to work better with longer documents, so let's try a corpus of all UN general assembly speeches that include the phrases "climate change" or "global warming"

In [11]:
# Load and analyze the climate speeches dataset (https://doi.org/10.7910/DVN/0TJX8Y)
climate_speeches <- read_csv("https://github.com/yse-eds-cert/yse-eds-cert-classroom-code-along-notebooks-code_along_notebooks/raw/refs/heads/main/course-4-environmental-analysis/module-13/data/climate_speeches.csv")

# Display basic information about the dataset
cat("Climate speeches dataset:\n")
cat("Number of speeches:", nrow(climate_speeches), "\n")
cat("Columns:", paste(names(climate_speeches), collapse = ", "), "\n\n")

# Function to tokenize, remove stop words, and stem text
process_speeches <- function(speech_data) {
  # Tokenize the speeches
  tokenized <- speech_data %>%
    unnest_tokens(word, content) %>%
    # Remove stop words
    anti_join(stop_words, by = "word") %>%
    # Remove numbers and single letters
    filter(!str_detect(word, "^\\d+$|^[a-z]$")) %>%
    # Stem the words
    mutate(word_stem = wordStem(word, language = "en"))

  return(tokenized)
}

# Process the climate speeches using the helper function
climate_speech_tokens <- process_speeches(climate_speeches)

# Calculate TF-IDF for climate speeches using word stems
climate_speech_tfidf <- climate_speech_tokens %>%
  count(filename, country,year,word_stem, sort = TRUE) %>%
  bind_tf_idf(word_stem, filename,  n)

# Display top TF-IDF terms
cat("Top TF-IDF terms in climate speeches:\n")
climate_speech_tfidf %>%
  arrange(desc(tf_idf)) %>%
  head(20) %>%
  select(filename, country, year, word_stem, n, tf_idf) %>%
  kable(col.names = c("Speech File", "Country", "Year", "Word Stem", "Count", "TF-IDF"))


Rows: 3031 Columns: 6
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): folder, filename, country, content
dbl (2): session, year

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Climate speeches dataset:
Number of speeches: 3031 
Columns: folder, filename, country, session, year, content 

Top TF-IDF terms in climate speeches:




|Speech File     |Country | Year|Word Stem    | Count|    TF-IDF|
|:---------------|:-------|----:|:------------|-----:|---------:|
|LKA_73_2018.txt |LKA     | 2018|sri          |    27| 0.2246337|
|ISL_65_2010.txt |ISL     | 2010|iceland      |    30| 0.2118816|
|MWI_47_1992.txt |MWI     | 1992|malawi       |    50| 0.2051991|
|LKA_73_2018.txt |LKA     | 2018|lanka        |    23| 0.1926896|
|GBR_78_2023.txt |GBR     | 2023|ai           |    30| 0.1860998|
|PAN_64_2009.txt |PAN     | 2009|panama       |    31| 0.1713661|
|AZE_75_2020.txt |AZE     | 2020|armenia      |    61| 0.1709739|
|MDA_79_2024.txt |MDA     | 2024|moldova      |    26| 0.1677749|
|ISL_67_2012.txt |ISL     | 2012|iceland      |    19| 0.1667625|
|PLW_61_2006.txt |PLW     | 2006|palau        |    26| 0.1655604|
|LAO_79_2024.txt |LAO     | 2024|pdr          |    24| 0.1650426|
|LKA_71_2016.txt |LKA     | 2016|sri          |    16| 0.1647389|
|TKM_79_2024.txt |TKM     | 2024|turkmenistan |    24| 0.1629176|
|SUR_62_

In [12]:
# Compare top TF-IDF words between Brazil and US in 2021
brazil_us_comparison <- climate_speech_tfidf %>%
  # Extract country and year from filename
  mutate(
    country = case_when(
      str_detect(filename, "BRA") ~ "Brazil",
      str_detect(filename, "USA") ~ "USA",
      TRUE ~ "Other"
    ),
    year = str_extract(filename, "\\d{4}")
  ) %>%
  # Filter for Brazil and USA in 2021
  filter(country %in% c("Brazil", "USA"), year == "2021") %>%
  # Get top 20 words for each country
  group_by(country) %>%
  slice_max(tf_idf, n = 20) %>%
  ungroup() %>%
  # Select relevant columns
  select(country, word_stem, tf_idf, n) %>%
  arrange(country, desc(tf_idf))

#get Brazil and USA words

brazil_words <- brazil_us_comparison %>% filter(country == "Brazil")
usa_words <- brazil_us_comparison %>% filter(country == "USA")
# Display top 20 words for Brazil
cat("\nBRAZIL - Top TF-IDF words:\n")
brazil_words %>%
  head(20) %>%
  kable(col.names = c("Country", "Word Stem", "TF-IDF", "Count"))

# Display top 20 words for USA
cat("\nUSA - Top TF-IDF words:\n")
usa_words %>%
  head(20) %>%
  kable(col.names = c("Country", "Word Stem", "TF-IDF", "Count"))




BRAZIL - Top TF-IDF words:




|Country |Word Stem     |    TF-IDF| Count|
|:-------|:-------------|---------:|-----:|
|Brazil  |brazil        | 0.0662784|    17|
|Brazil  |brazilian     | 0.0295051|     4|
|Brazil  |auction       | 0.0291792|     3|
|Brazil  |amazon        | 0.0232887|     4|
|Brazil  |railroad      | 0.0165894|     2|
|Brazil  |vaccin        | 0.0164732|     5|
|Brazil  |contract      | 0.0143807|     3|
|Brazil  |billion       | 0.0138804|     7|
|Brazil  |railway       | 0.0132264|     2|
|Brazil  |1500s         | 0.0125260|     1|
|Brazil  |concess       | 0.0123097|     2|
|Brazil  |indigen       | 0.0114586|     3|
|Brazil  |doctor        | 0.0110250|     2|
|Brazil  |equival       | 0.0110250|     2|
|Brazil  |administr     | 0.0109578|     4|
|Brazil  |biom          | 0.0108094|     1|
|Brazil  |dictatorship’ | 0.0108094|     1|
|Brazil  |sector’       | 0.0108094|     1|
|Brazil  |cent          | 0.0105149|     9|
|Brazil  |5g            | 0.0103599|     1|


USA - Top TF-IDF words:




|Country |Word Stem   |    TF-IDF| Count|
|:-------|:-----------|---------:|-----:|
|USA     |alli        | 0.0116555|     8|
|USA     |dose        | 0.0101639|     5|
|USA     |covid       | 0.0099363|     9|
|USA     |vaccin      | 0.0070757|     6|
|USA     |tool        | 0.0069653|     8|
|USA     |defend      | 0.0064269|     7|
|USA     |inflect     | 0.0057341|     2|
|USA     |dna         | 0.0056736|     2|
|USA     |stamp       | 0.0052006|     2|
|USA     |american    | 0.0050114|     5|
|USA     |tip         | 0.0049281|     2|
|USA     |cyberattack | 0.0048983|     2|
|USA     |shot        | 0.0048409|     2|
|USA     |seek        | 0.0047434|    12|
|USA     |richer      | 0.0046605|     2|
|USA     |answer      | 0.0046572|     4|
|USA     |jewish      | 0.0046370|     2|
|USA     |variant     | 0.0045692|     2|
|USA     |pandem      | 0.0045377|     6|
|USA     |agnost      | 0.0044836|     1|

### Creating a Document-Term Matrix (DTM)

For topic modeling, we need to convert our data into a Document-Term Matrix format where:
- Rows represent documents
- Columns represent terms
- Values represent term frequencies (or TF-IDF weights)


In [13]:
# Create Document-Term Matrix for topic modeling
# First, let's examine term frequencies to identify common and uncommon words
term_frequencies <- climate_speech_tokens %>%
  count(word_stem, sort = TRUE)

# Calculate document frequencies (how many documents each term appears in)
doc_frequencies <- climate_speech_tokens %>%
  distinct(filename, word_stem) %>%
  count(word_stem, sort = TRUE)

# Get total number of documents
total_docs <- climate_speech_tokens %>%
  distinct(filename) %>%
  nrow()

# Filter out very common words (appearing in >80% of documents) and very rare words (appearing in <2 documents)
filtered_terms <- doc_frequencies %>%
  filter(n >= 2 & n <= (total_docs * 0.8)) %>%
  pull(word_stem)

# Create filtered DTM
climate_dtm <- climate_speech_tokens %>%
  filter(word_stem %in% filtered_terms) %>%
  count(filename, word_stem) %>%
  cast_dtm(filename, word_stem, n)

# Examine the DTM
cat("Original vocabulary size:", nrow(term_frequencies), "\n")
cat("Filtered vocabulary size:", length(filtered_terms), "\n")
cat("DTM dimensions:", dim(climate_dtm), "\n")
cat("Number of documents:", nrow(climate_dtm), "\n")
cat("Number of terms:", ncol(climate_dtm), "\n")

# Look at the first few terms and documents
inspect(climate_dtm[1:5, 1:10])


Original vocabulary size: 22109 
Filtered vocabulary size: 13955 
DTM dimensions: 3031 13955 
Number of documents: 3031 
Number of terms: 13955 
<<DocumentTermMatrix (documents: 5, terms: 10)>>
Non-/sparse entries: 31/19
Sparsity           : 38%
Maximal term length: 12
Weighting          : term frequency (tf)
Sample             :
                 Terms
Docs              85,000 accept access activ adopt afghan afghanistan
  AFG_62_2007.txt      1      1      1     1     1      5          22
  AFG_63_2008.txt      0      0      0     0     0      6          13
  AFG_71_2016.txt      0      0      1     3     0     10          30
  AFG_73_2018.txt      0      1      0     2     3      5          15
  AFG_74_2019.txt      0      1      0     1     0     36          14
                 Terms
Docs              afghanistan’ ago ahead
  AFG_62_2007.txt            3   4     1
  AFG_63_2008.txt            4   0     0
  AFG_71_2016.txt            3   0     0
  AFG_73_2018.txt            3   0    

## Part 3: Topic Modeling with Latent Dirichlet Allocation (LDA)

### What is Topic Modeling?

**Topic modeling** is an unsupervised machine learning technique that discovers abstract topics in a collection of documents.

**LDA (Latent Dirichlet Allocation)** assumes:
- Each document is a mixture of topics
- Each topic is a mixture of words
- The algorithm finds the optimal topic-word and document-topic distributions

### Fitting an LDA Model

We'll start by fitting a model with 5 topics to see what themes emerge from our climate data.


In [ ]:
# Fit LDA model with 5 topics
set.seed(123)  # For reproducibility
climate_lda <- LDA(climate_dtm, k = 5, control = list(seed = 123))

# Extract the top terms for each topic
climate_topics <- tidy(climate_lda, matrix = "beta")

# Look at the top 10 terms for each topic
top_terms <- climate_topics %>%
  group_by(topic) %>%
  top_n(10, beta) %>%
  ungroup() %>%
  arrange(topic, -beta)

# Display the results
top_terms %>%
  kable(col.names = c("Topic", "Term", "Beta (Probability)"))


In [ ]:
# Visualize the top terms for each topic
top_terms <- climate_topics %>%
  group_by(topic) %>%
  top_n(10, beta) %>%
  ungroup() %>%
  arrange(topic, -beta)
top_terms %>%
  mutate(term = reorder_within(term, beta, topic)) %>%
  ggplot(aes(term, beta, fill = factor(topic))) +
  geom_col(show.legend = FALSE) +
  facet_wrap(~ topic, scales = "free") +
  coord_flip() +
  scale_x_reordered() +
  labs(title = "Top 10 Terms in Each Topic",
       x = "Terms",
       y = "Beta (Topic-Term Probability)") +
  theme_minimal()


### Interpreting Topic Prevalence

Let's examine how prevalent each topic is across our documents


In [ ]:
# Extract document-topic probabilities (gamma)
doc_topics <- tidy(climate_lda, matrix = "gamma")
topic_features <- doc_topics %>%
  select(document, topic, gamma) %>%
  pivot_wider(names_from = topic, names_prefix = "topic_", values_from = gamma) %>%
  left_join(climate_speeches %>% select(filename, country, year), by = c("document" = "filename")) %>%
  select(-document)
# Calculate average topic prevalence
topic_prevalence <- doc_topics %>%
  group_by(topic) %>%
  summarise(avg_gamma = mean(gamma)) %>%
  arrange(desc(avg_gamma))

# Display topic prevalence
topic_prevalence %>%
  ggplot(aes(x = reorder(factor(topic), avg_gamma), y = avg_gamma)) +
  geom_col(fill = "darkgreen", alpha = 0.8) +
  coord_flip() +
  labs(title = "Average Topic Prevalence Across All Documents",
       x = "Topic",
       y = "Average Gamma (Document-Topic Probability)") +
  theme_minimal()


In [ ]:
# Create topic features dataset by reshaping gamma values
topic_features <- doc_topics %>%
  select(document, topic, gamma) %>%
  pivot_wider(names_from = topic, names_prefix = "topic_", values_from = gamma) %>%
  left_join(climate_speeches %>% select(filename, country, year), by = c("document" = "filename")) %>%
  select(-document)

# Display the first few rows
head(topic_features) %>%
  kable()


# Analyze topic evolution over time using topic_features
topic_time_evolution <- topic_features %>%
  filter(!is.na(year)) %>%
  group_by(year) %>%
  summarise(
    topic_1 = mean(topic_1, na.rm = TRUE),
    topic_2 = mean(topic_2, na.rm = TRUE),
    topic_3 = mean(topic_3, na.rm = TRUE),
    topic_4 = mean(topic_4, na.rm = TRUE),
    topic_5 = mean(topic_5, na.rm = TRUE),
    .groups = "drop"
  )

# Reshape for visualization
topic_time_long <- topic_time_evolution %>%
  pivot_longer(cols = starts_with("topic_"),
               names_to = "topic",
               values_to = "avg_proportion")

# Visualize topic evolution over time
topic_time_long %>%
  ggplot(aes(x = year, y = avg_proportion, color = topic)) +
  geom_line(size = 1.2) +
  geom_point(size = 2) +
  labs(title = "Topic Prevalence Evolution Over Time",
       x = "Year",
       y = "Average Topic Proportion",
       color = "Topic") +
  theme_minimal() +
  scale_color_brewer(type = "qual", palette = "Set1")

# Show the data in table format
topic_time_evolution %>%
  kable(digits = 4, caption = "Average Topic Proportions by Year")


## Exercise: Explore Different Numbers of Topics

Now it's your turn to experiment! Try fitting LDA models with different numbers of topics (k) and see how it affects the model performance.


In [ ]:
# Exercise: Run a topic model on the climate tweets dataset with 5 topics.

# YOUR CODE HERE

In [ ]:
# Exercise: Try different numbers of topics
# YOUR CODE HERE

# Example: Fit models with k = 3, 7, 10 topics
# Do you get similar results?



## Summary

In this lesson, we've covered advanced text analysis techniques using TF-IDF and topic modeling:

1. **TF-IDF Analysis**: We calculated term frequency-inverse document frequency to identify words that are particularly important to specific documents

2. **Document-Term Matrix**: We constructed a DTM as the foundation for topic modeling

3. **Topic Modeling with LDA**: We fitted a Latent Dirichlet Allocation model to discover latent topics in climate change discussions

4. **Topic Interpretation**: We examined the most probable words for each topic and analyzed topic prevalence

### Key Insights:
- Topic models provide interpretable representations of document collections
- TF-IDF helps identify distinctive vocabulary across documents
- Topic proportions can serve as meaningful features for prediction tasks
- The choice of number of topics (k) affects both interpretability and predictive performance

These techniques are particularly valuable for:
- **Policy Analysis**: Understanding themes in policy documents
- **Social Media Monitoring**: Tracking public discourse on environmental issues
- **Content Classification**: Automatically categorizing environmental texts
- **Trend Analysis**: Monitoring how environmental topics evolve over time
